# Framework Comparison: DSPy vs LangGraph vs Pydantic AI vs OpenAI Agents SDK vs CrewAI (Chapter 8)

This notebook accompanies Chapter 8 §8.2.3 of *Context Engineering with DSPy*. The same agent — a question-answerer with a knowledge-base search tool and a calculator — is built five times so you can compare the surface area of each framework side by side.

**Required environment variable**

- `OPENAI_API_KEY` — every framework here calls an OpenAI model.

**Per-framework installs**

Each non-DSPy section starts with its own `%pip install` cell so you can run a single framework in isolation. These packages intentionally do **not** live in `../requirements.txt` because most readers will only want one of them.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5-mini"))

## Shared tools

All five agents need a knowledge-base search and a calculator. We define plain Python versions here; each framework section re-defines them in whatever style that framework expects.

In [ ]:
def search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"DSPy notes for: {query}"


def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))

## 1. DSPy

Five lines including the call. The tool definitions are the plain Python functions above.

In [ ]:
agent = dspy.ReAct(
    "question -> answer",
    tools=[search_knowledge_base, calculate],
    max_iters=5,
)
result = agent(question="What is DSPy and what is 15 * 23?")
print(result.answer)

## 2. LangGraph

LangGraph ships a `create_react_agent` shortcut. Tools need the `@tool` decorator, you set up the chat model yourself, and you work with message dicts instead of a typed signature.

The install cell below is isolated to this section.

In [ ]:
%pip install langgraph langchain-openai -q

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool


@tool
def search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"DSPy notes for: {query}"


@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


model = ChatOpenAI(model="gpt-4o-mini")
agent = create_react_agent(model, [search_knowledge_base, calculate])
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is DSPy and what is 15 * 23?"}]}
)
print(result["messages"][-1].content)

## 3. Pydantic AI

Pydantic AI registers tools via decorators bound to a specific agent. Output is unstructured text by default — no signature system.

The install cell below is isolated to this section.

In [ ]:
%pip install pydantic_ai -q

In [ ]:
from pydantic_ai import Agent

agent = Agent("openai:gpt-4o-mini")


@agent.tool_plain
def search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"DSPy notes for: {query}"


@agent.tool_plain
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


result = agent.run_sync("What is DSPy and what is 15 * 23?")
print(result.output)

## 4. OpenAI Agents SDK

Decorator-based tools, an `Agent` object, and a `Runner` to execute. You are locked into OpenAI models and there is no built-in optimization path.

The install cell below is isolated to this section.

In [ ]:
%pip install openai-agents -q

In [ ]:
from agents import Agent, Runner, function_tool


@function_tool
def search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"DSPy notes for: {query}"


@function_tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


agent = Agent(
    name="QA Agent",
    instructions="Answer questions using available tools.",
    tools=[search_knowledge_base, calculate],
)
result = Runner.run_sync(agent, "What is DSPy and what is 15 * 23?")
print(result.final_output)

## 5. CrewAI

CrewAI is designed for multi-agent orchestration, so even a single agent requires a role, goal, backstory, task object, and crew wrapper.

The install cell below is isolated to this section.

In [ ]:
%pip install crewai -q

In [ ]:
from crewai import Agent, Task, Crew
from crewai.tools import tool


@tool("Search Knowledge Base")
def search_knowledge_base(query: str) -> str:
    """Search our product documentation for relevant information."""
    return f"DSPy notes for: {query}"


@tool("Calculator")
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    return str(eval(expression))


agent = Agent(
    role="Research Assistant",
    goal="Answer questions accurately using available tools",
    backstory="You are a helpful assistant.",
    tools=[search_knowledge_base, calculate],
)
task = Task(
    description="What is DSPy and what is 15 * 23?",
    expected_output="A clear answer",
    agent=agent,
)
crew = Crew(agents=[agent], tasks=[task])
result = crew.kickoff()
print(result)